---
# **UCCD3133: Assignment 2**
---

## **Group Information**

**Group Number:** 29

> **Member 1:** Brian Lee Zhen Hui (2206413)
>
> **Member 2:** Lee Cheng Jun (2206342)
>
> **Member 3:** Ng Che Te (2206349)

---

## **Work Distribution**

* **Ingestion (Section 1 —  PDF loading, metadata tagging, chunking, Chroma indexing):** Brian Lee Zhen Hui

<br>

* **RAG Workflow —  Query Transformation & Retrieval (Section 2 — HyDE, Multi-Query Retriever, Metadata Filtering):** Lee Cheng Jun

<br>

* **RAG Workflow — Compression, Chain & Chat Interface (Section 3 — Dual-stage compression, guardrail, ConversationalRetrievalChain, ipywidgets UI):** Ng Che Te

---

## **Heading Structure of Source Document(s)**

**Document:** Progress towards the Sustainable Development Goals – Report of the Secretary-General 2024  
**Publisher:** United Nations  
**Licence:** Public  
**URL:** https://unstats.un.org/sdgs/files/report/2024/SG-SDG-Progress-Report-2024-advanced-unedited-version.pdf


```
Goal 1: No Poverty
Goal 2: Zero Hunger
Goal 3: Good Health and Well-being
Goal 4: Quality Education
Goal 5: Gender Equality
Goal 6: Clean Water and Sanitation
Goal 7: Affordable and Clean Energy
Goal 8: Decent Work and Economic Growth
Goal 9: Industry, Innovation and Infrastructure
Goal 10: Reduced Inequalities
Goal 11: Sustainable Cities and Communities
Goal 12: Responsible Consumption and Production
Goal 13: Climate Action
Goal 14: Life Below Water
Goal 15: Life on Land
Goal 16: Peace, Justice and Strong Institutions
Goal 17: Partnerships for the Goals
```

**Enriched metadata:**

```python
metadata={'source': 'UN SDG Progress Report 2024', 'goal': 'Goal 3', 'topic': 'Good Health and Well-being'}
```

---

## **Modifications & New Components**




### **Modifications**




#### **Modification 1 — Section‑Aware Metadata Tagging**
**(Section 1, Brian)**

During ingestion, each document page is scanned for domain‑specific keywords to assign `goal` and `topic` metadata labels. The lab baseline (Lab 6) used no metadata. This modification enables downstream goal‑level metadata filtering in Section 2, narrowing the vector search space to only the most relevant document region and reducing retrieval noise.


<br>

#### **Modification 2 — Multi‑Query Retriever with Domain‑Tuned Prompt**
**(Section 2, Lee Cheng Jun)**

The LangChain `MultiQueryRetriever` is configured with a custom SDG‑focused prompt that forces three distinct query angles: (1) statistical/numerical, (2) policy/programme, and (3) regional/country comparison. The generic LangChain default generates plain paraphrases. This domain‑specific approach produces more useful query diversity for SDG data retrieval, improving recall across diverse chunk types.


<br>

### **New Components**




#### **New Component 1 — HyDE Query Rewriting**
**(Section 2, Lee Cheng Jun)**

Before retrieval, the LLM generates a hypothetical UN‑style passage that represents what an ideal answer would look like. This passage replaces the raw user query for embedding‑based retrieval. Because the hypothetical passage is semantically similar in structure and vocabulary to actual document chunks, it closes the semantic gap between short conversational queries and long, formal report paragraphs — improving retrieval accuracy for dense sustainable development data.


<br>

#### **New Component 2 — Dual‑Stage Context Compression Pipeline**
**(Section 3, Ng Che Te)**

A `DocumentCompressorPipeline` chains two stages: (1) `EmbeddingsRedundantFilter` removes near‑duplicate chunks using cosine similarity before the expensive LLM step; (2) `LLMChainExtractor` then extracts only sentences directly relevant to the query. Lab 7 used only a single `LLMChainExtractor`. The dual pipeline reduces both redundancy and token cost, delivering a tighter, higher‑quality context to the final generation step.


<br>

#### **New Component 3 — Out‑of‑Scope Query Guardrail**
**(Section 3, Ng Che Te)**

A lightweight LLM relevance classifier evaluates every user query before invoking the full RAG pipeline. If the query is unrelated to the UN SDG Progress Report, a polite refusal is returned immediately, meeting the assignment requirement that "the assistant will not entertain any other irrelevant queries" while also preventing unnecessary API usage and hallucinated off‑topic answers.

---------------------------------------------------------
## **Ingestion**
---------------------------------------------------------


### Section 1: Ingestion & Chroma Indexing
**Responsible: Brian Lee Zhen Hui**

| Component | Description | Status |
|-----------|-------------|--------|
| Package & API setup | Installs LangChain/OpenAI dependencies and retrieves API key from Colab Secrets. | Reused |
| PDF download | Downloads UN Secretary-General SDG Progress Report 2024 from UN Statistics Division. | Reused |
| PDF loading | Loads document using `PyPDFLoader`. | Reused |
| Metadata tagging | Uses scored keyword matching to assign `goal`/`topic` labels for each page, enabling goal-level filtering. | **Modified** |
| Chunking strategy | `RecursiveCharacterTextSplitter` with `chunk_size=800`, `chunk_overlap=150` for focused chunks while preserving cross-boundary context (Lab baseline used 1000/0). | **Modified** |
| Embeddings & Chroma | OpenAI `text-embedding-3-small` embeddings persisted to Chroma vector store for similarity search. | Reused |


In [ ]:
# Install all required packages
%%capture
!pip install -q \
    langchain langchain-community langchain-openai langchain-chroma \
    langchain-text-splitters \
    chromadb openai tiktoken pypdf ipywidgets \
    'requests==2.32.4'

In [ ]:
import os
from google.colab import userdata

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY2")

print("API key configured.")

API key configured.


In [ ]:
# Download source PDF: UN Secretary-General SDG Progress Report 2024

import urllib.request, os

PDF_URL  = "https://unstats.un.org/sdgs/files/report/2024/SG-SDG-Progress-Report-2024-advanced-unedited-version.pdf"
PDF_PATH = "A2_29_source.pdf"

if not os.path.exists(PDF_PATH):
    print("Downloading UN SDG Progress Report 2024...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print(f"Downloaded: {PDF_PATH}")
else:
    print(f"PDF already exists: {PDF_PATH}")

print(f"File size: {os.path.getsize(PDF_PATH) / 1024:.1f} KB")

Downloaded: A2_29_source.pdf
File size: 505.9 KB


In [ ]:
# 1.1  Load PDF

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

print("Loading PDF...")
loader    = PyPDFLoader(PDF_PATH)
raw_pages = loader.load()
print(f"Loaded {len(raw_pages)} pages from the PDF.")

Loading PDF...
Loaded 26 pages from the PDF.


In [ ]:
# 1.2  Section-Aware Metadata Tagging

GOAL_MAP = {
    "poverty": ("Goal 1", "No Poverty"),
    "extreme poverty": ("Goal 1", "No Poverty"),
    "hunger": ("Goal 2", "Zero Hunger"),
    "food security": ("Goal 2", "Zero Hunger"),
    "malnutrition": ("Goal 2", "Zero Hunger"),
    "health": ("Goal 3", "Good Health and Well-being"),
    "maternal": ("Goal 3", "Good Health"),
    "child mortality": ("Goal 3", "Good Health"),
    "hiv": ("Goal 3", "Good Health"),
    "tuberculosis": ("Goal 3", "Good Health"),
    "malaria": ("Goal 3", "Good Health"),
    "education": ("Goal 4", "Quality Education"),
    "gender": ("Goal 5", "Gender Equality"),
    "women": ("Goal 5", "Gender Equality"),
    "water": ("Goal 6", "Clean Water and Sanitation"),
    "sanitation": ("Goal 6", "Clean Water and Sanitation"),
    "energy": ("Goal 7", "Affordable and Clean Energy"),
    "economic growth": ("Goal 8", "Decent Work and Economic Growth"),
    "employment": ("Goal 8", "Decent Work"),
    "infrastructure": ("Goal 9", "Industry, Innovation and Infrastructure"),
    "inequalit": ("Goal 10", "Reduced Inequalities"),
    "cities": ("Goal 11", "Sustainable Cities and Communities"),
    "consumption": ("Goal 12", "Responsible Consumption and Production"),
    "climate": ("Goal 13", "Climate Action"),
    "ocean": ("Goal 14", "Life Below Water"),
    "marine": ("Goal 14", "Life Below Water"),
    "biodiversity": ("Goal 15", "Life on Land"),
    "forest": ("Goal 15", "Life on Land"),
    "peace": ("Goal 16", "Peace, Justice and Strong Institutions"),
    "justice": ("Goal 16", "Peace, Justice"),
    "partnership": ("Goal 17", "Partnerships for the Goals"),
    "sdg": ("General", "SDG Progress"),
}

def detect_goal_metadata(page_content: str) -> dict:
    content_lower = page_content.lower()
    scores = {}

    for keyword, (goal, topic) in GOAL_MAP.items():
        if keyword in content_lower:
            scores[(goal, topic)] = scores.get((goal, topic), 0) + content_lower.count(keyword)

    if not scores:
        return {"goal": "General", "topic": "SDG Progress"}

    best_goal, best_topic = max(scores, key=scores.get)
    return {"goal": best_goal, "topic": best_topic}


for page in raw_pages:
    meta = detect_goal_metadata(page.page_content)
    page.metadata.update(meta)
    page.metadata["source"] = "UN SDG Progress Report 2024"

print("Metadata tagging complete. Sample metadata:")
print(raw_pages[5].metadata)

Metadata tagging complete. Sample metadata:
{'producer': 'Adobe PDF Library 24.2', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2024-06-05T15:01:31-04:00', 'agenda title1': 'High-level political forum on sustainable development, convened under the auspices of the Economic and Social Council', 'agenda1': 'Agenda item 6', 'author': 'Argenis Santana', 'comment': '', 'comments': '', 'company': '', 'contenttypeid': '0x01010021AC74FF7B22304EB62C1AD387B8ECBC', 'doctype': 'S', 'draftpages': '', 'grammarlydocumentid': '8f26d7122306b7200259affc103a3baf959a043562a249cffff8dda9349bd31f', 'jobno': '2206472', 'keywords': '', 'language': 'English', 'mediaserviceimagetags': '', 'moddate': '2024-06-05T15:01:51-04:00', 'odsrefjobno': '2233513E', 'oecddocumentid': '86051959920F3EFFBDCC7E897CF7E8CBEA5ADB7AE5A843816435E5CB32F63F6F', 'oecddocumentcotelanghash': '', 'operator': '', 'session1': '2022 session', 'sourcemodified': '', 'subject': '', 'symbol1': 'E/2022/55', 'symbol2': '', 'title': 

In [ ]:
# 1.3  Chunking Strategy
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(raw_pages)
avg_len = sum(len(c.page_content) for c in chunks) // len(chunks)

print(f"Created {len(chunks)} chunks from {len(raw_pages)} pages.")
print(f"Average chunk size: {avg_len} characters")
print(f"\nSample chunk metadata : {chunks[10].metadata}")
print(f"Sample chunk content  :\n{chunks[10].page_content[:300]}...")

Created 135 chunks from 26 pages.
Average chunk size: 707 characters

Sample chunk metadata : {'producer': 'Adobe PDF Library 24.2', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2024-06-05T15:01:31-04:00', 'agenda title1': 'High-level political forum on sustainable development, convened under the auspices of the Economic and Social Council', 'agenda1': 'Agenda item 6', 'author': 'Argenis Santana', 'comment': '', 'comments': '', 'company': '', 'contenttypeid': '0x01010021AC74FF7B22304EB62C1AD387B8ECBC', 'doctype': 'S', 'draftpages': '', 'grammarlydocumentid': '8f26d7122306b7200259affc103a3baf959a043562a249cffff8dda9349bd31f', 'jobno': '2206472', 'keywords': '', 'language': 'English', 'mediaserviceimagetags': '', 'moddate': '2024-06-05T15:01:51-04:00', 'odsrefjobno': '2233513E', 'oecddocumentid': '86051959920F3EFFBDCC7E897CF7E8CBEA5ADB7AE5A843816435E5CB32F63F6F', 'oecddocumentcotelanghash': '', 'operator': '', 'session1': '2022 session', 'sourcemodified': '', 'subject': ''

In [ ]:
# 1.4  Embeddings & Chroma Vector Stor

CHROMA_PERSIST_DIR = "./sdg_chroma_db"

print("Initialising OpenAI embeddings (text-embedding-3-small)...")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Building Chroma vector store (may take a minute)...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PERSIST_DIR,
    collection_name="sdg_progress_2024",
)

print(f"Chroma vector store built and persisted at '{CHROMA_PERSIST_DIR}'")
print(f"Total documents indexed: {vectorstore._collection.count()}")

Initialising OpenAI embeddings (text-embedding-3-small)...
Building Chroma vector store (may take a minute)...
Chroma vector store built and persisted at './sdg_chroma_db'
Total documents indexed: 135


In [ ]:
# 1.5  Sanity Check: Basic Similarity Search
test_query   = "What is the global extreme poverty rate?"
test_results = vectorstore.similarity_search(test_query, k=2)

print(f"Test query: '{test_query}'")
for i, doc in enumerate(test_results):
    print(f"\n  Result {i+1} | Goal: {doc.metadata.get('goal')} | Page: {doc.metadata.get('page')}")
    print(f"  {doc.page_content[:200]}...")

Test query: 'What is the global extreme poverty rate?'

  Result 1 | Goal: Goal 1 | Page: 5
  A/79/79 
E/2022/55  
 
 6/26 
 
increasingly out of reach, particularly in regions that lack the fiscal capacity to cope 
with economic stresses . 
• Target 1.1 :  
o Extreme poverty levels returned t...

  Result 2 | Goal: Goal 1 | Page: 5
  decreased , from 8.4% i n 2015 to 6.9% in 2023 . However, nearly 241 million 
workers globally were still living in extreme poverty in 2023 and little 
positive change is expected in 2024.  
• 
Target...


--------------------------------------------------------
## **RAG Workflow**
--------------------------------------------------------


### Section 2: Query Transformation & Advanced Retrieval
**Responsible: Lee Cheng Jun**

| Component | Description | Status |
|-----------|-------------|--------|
| HyDE (Hypothetical Document Embeddings) | Rewrites the user query as a UN-style hypothetical passage before retrieval, reducing the semantic gap between conversational queries and dense report text. | **New** |
| Multi-Query Retriever with domain-tuned prompt | Uses a custom prompt to generate 3 query variants from statistical, policy, and regional angles instead of generic paraphrases. | **Modified** |
| Metadata Filtering | Uses Chroma `where` filtering to restrict retrieval to the detected SDG goal, reducing off-goal noise. | **New** |
| Composed Pipeline | HyDE -> MultiQuery -> Metadata-filtered retrieval -> Deduplication, with priority given to goal-matching chunks to reduce noisy retrieval. | **New** |


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.retrievers import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from typing import List
import logging

# Suppress verbose MultiQueryRetriever logs during demo
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM (gpt-4o-mini) initialised.")

LLM (gpt-4o-mini) initialised.


In [ ]:
# 2.1  HyDE: Hypothetical Document Embeddings

HYDE_PROMPT = ChatPromptTemplate.from_template(
    "You are an expert in sustainable development and United Nations SDG reports.\n"
    "A user asked the following question about the UN Secretary-General SDG Progress Report 2024:\n\n"
    "Question: {question}\n\n"
    "Write a short, factual paragraph (3-5 sentences) that would appear in that report "
    "as an answer to this question. Use formal, technical UN publication language. "
    "Do NOT invent specific numbers — write in the style of the document.\n"
    "Hypothetical passage:"
)

hyde_chain = HYDE_PROMPT | llm | StrOutputParser()

def hyde_query_transform(question: str) -> str:
    return hyde_chain.invoke({"question": question})

# Test HyDE
test_q   = "What progress has been made on reducing extreme poverty?"
hyde_out = hyde_query_transform(test_q)
print(f"Original query: {test_q}")
print(f"\nHypothetical passage:\n{hyde_out}")

Original query: What progress has been made on reducing extreme poverty?

Hypothetical passage:
As of 2024, significant challenges remain in the global effort to reduce extreme poverty, particularly in the wake of the COVID-19 pandemic, which has exacerbated existing vulnerabilities. The latest data indicate that the number of people living in extreme poverty has increased for the first time in over two decades, reversing years of progress. Efforts to implement social protection measures and promote inclusive economic growth are critical to addressing the needs of the most marginalized populations. Continued international cooperation and investment in sustainable development initiatives are essential to achieve the target of eradicating extreme poverty in all its forms by 2030.


In [ ]:
# 2.2  Multi-Query Retriever with Domain-Tuned Prompt

class LineListOutputParser(BaseOutputParser):
    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return [l.strip().lstrip("0123456789.-) ") for l in lines if l.strip()]

MULTI_QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template=(
        "You are a sustainable development research assistant searching the UN Secretary‑General SDG Progress Report 2024.\n"
        "Generate exactly 3 different search queries that approach the user's question from different angles:\n"
        "1. Statistical/numerical angle\n"
        "2. Policy/programme angle\n"
        "3. Regional/country comparison angle\n\n"
        "Original question: {question}\n\n"
        "Output ONLY the 3 queries, one per line, no numbering or bullet points:"
    )
)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

multi_query_retriever = MultiQueryRetriever(
    retriever=base_retriever,
    llm_chain=MULTI_QUERY_PROMPT | llm | LineListOutputParser(),
    parser_key="lines",
    include_original=True,
)

print("Multi-Query Retriever configured with domain-tuned prompt.")

Multi-Query Retriever configured with domain-tuned prompt.


In [ ]:
# 2.3  Metadata Filtering

def get_filtered_retriever(goal: str, k: int = 4):
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k, "filter": {"goal": goal}},
    )

GOAL_KEYWORDS = {
    "poverty": "Goal 1",
    "hunger": "Goal 2", "food": "Goal 2",
    "health": "Goal 3", "maternal": "Goal 3", "child mortality": "Goal 3", "hiv": "Goal 3", "tb": "Goal 3", "malaria": "Goal 3",
    "education": "Goal 4",
    "gender": "Goal 5", "women": "Goal 5",
    "water": "Goal 6", "sanitation": "Goal 6",
    "energy": "Goal 7",
    "economic": "Goal 8", "employment": "Goal 8",
    "infrastructure": "Goal 9", "industry": "Goal 9",
    "inequalit": "Goal 10",
    "cities": "Goal 11",
    "consumption": "Goal 12", "production": "Goal 12",
    "climate": "Goal 13",
    "ocean": "Goal 14", "marine": "Goal 14",
    "biodiversity": "Goal 15", "forest": "Goal 15", "land": "Goal 15",
    "peace": "Goal 16", "justice": "Goal 16",
    "partnership": "Goal 17",
}

def get_goal_for_query(query: str):
    q = query.lower()
    for kw, gl in GOAL_KEYWORDS.items():
        if kw in q:
            return gl
    return None

# Test metadata filtering
fret  = get_filtered_retriever("Goal 3", k=2)
fdocs = fret.invoke("maternal mortality rates")
print(f"Metadata filter test — retrieved {len(fdocs)} docs restricted to Goal 3:")
for d in fdocs:
    print(f"  Goal: {d.metadata.get('goal')} | Topic: {d.metadata.get('topic')}")

Metadata filter test — retrieved 2 docs restricted to Goal 3:
  Goal: Goal 3 | Topic: Good Health
  Goal: Goal 3 | Topic: Good Health


In [ ]:
# 2.4  Composed Advanced Retrieval Pipeline

def advanced_retrieve(query: str) -> List:
    # Step 1: HyDE - transform user query into a hypothetical UN-style passage
    hyde_passage = hyde_query_transform(query)

    # Step 2: Multi-query retrieval on the HyDE-transformed passage
    docs_mq = multi_query_retriever.invoke(hyde_passage)

    # Step 3: Goal-filtered retrieval
    goal = get_goal_for_query(query)
    docs_filtered = []
    if goal:
        docs_filtered = get_filtered_retriever(goal, k=3).invoke(query)

    # Step 4: Merge and deduplicate
    seen, unique = set(), []
    for doc in (docs_mq + docs_filtered):
        key = doc.page_content[:100]
        if key not in seen:
            seen.add(key)
            unique.append(doc)

    if goal:
        goal_docs = [d for d in unique if d.metadata.get("goal") == goal]
        general_docs = [d for d in unique if d.metadata.get("goal") == "General"]
        unique = goal_docs


    return unique[:4]



# Test composed pipeline
docs = advanced_retrieve("What progress has been made on reducing global poverty?")
print(f"Advanced retrieval returned {len(docs)} unique documents.")
for d in docs:
    print(f"  Goal: {d.metadata.get('goal')} | Pg: {d.metadata.get('page_label', d.metadata.get('page'))} | {d.page_content}")


Advanced retrieval returned 3 unique documents.
  Goal: Goal 1 | Pg: 6 | A/79/79 
E/2022/55  
 
 6/26 
 
increasingly out of reach, particularly in regions that lack the fiscal capacity to cope 
with economic stresses . 
• Target 1.1 :  
o Extreme poverty levels returned to pre -pandemic levels in most countries by 
2022, except in low -income countries where recovery has been slower. In 
2022, 9% of the world's  population or 712 million people were living in 
extreme poverty, an increase of 23 m illion people compared to 2019. If 
current trend s continue , 590 million people , or 6.9% of the world’s 
population  will still live in extreme poverty by 2030.   
o The share of the world’s working population living in poverty has steadily 
decreased , from 8.4% i n 2015 to 6.9% in 2023 . However, nearly 241 million
  Goal: Goal 1 | Pg: 5 | since 2015) , comparing the 2019 database  and 2024 database , by Goal (percentage) 
 
 
Goal 1. End poverty in all its forms everywhere  
18. Global 

### Section 3: Context Compression, RAG Chain & Guardrail
**Responsible: Ng Che Te**

| Component | Description | Status |
|-----------|-------------|--------|
| Dual‑stage compression pipeline | `EmbeddingsRedundantFilter` removes near‑duplicate chunks via cosine similarity; `LLMChainExtractor` extracts only query‑relevant sentences. | **New** |
| Out‑of‑scope query guardrail | Lightweight LLM classifier evaluates every query and refuses irrelevant ones before invoking the full RAG pipeline, preventing hallucination and saving API cost. | **New** |
| ConversationalRetrievalChain | Uses `ConversationBufferWindowMemory` (k=5) for multi‑turn context with standalone‑question condensation. | Reused |

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor, DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document

# 3.1  Dual-Stage Compression Pipeline

redundancy_filter   = EmbeddingsRedundantFilter(embeddings=embeddings)
llm_extractor       = LLMChainExtractor.from_llm(llm)
compressor_pipeline = DocumentCompressorPipeline(
    transformers=[redundancy_filter, llm_extractor]
)

class AdvancedRetrieverAdapter(BaseRetriever):
    model_config = {"arbitrary_types_allowed": True}

    def _get_relevant_documents(self, query: str) -> List[Document]:
        return advanced_retrieve(query)
    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor_pipeline,
    base_retriever=AdvancedRetrieverAdapter(),
)

print("Dual-stage compression retriever configured.")

Dual-stage compression retriever configured.


In [ ]:
# 3.2  Out-of-Scope Query Guardrail

GUARDRAIL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a relevance classifier. Determine if the user's question is related to "
     "the UN SDG Progress Report 2024, which covers:\n"
     "- 17 Sustainable Development Goals (poverty, hunger, health, education, gender, water, energy, etc.)\n"
     "- Global and regional statistics on SDG indicators\n"
     "- Policy challenges and progress towards the 2030 Agenda\n\n"
     "Reply with ONLY 'RELEVANT' or 'IRRELEVANT'. No other text."),
    ("human", "Question: {question}")
])

guardrail_chain = GUARDRAIL_PROMPT | llm | StrOutputParser()

OUT_OF_SCOPE_REPLY = (
    "I'm sorry, I can only answer questions related to the "
    "UN SDG Progress Report 2024. Please ask about topics such as poverty, hunger, health, "
    "education, gender equality, climate action, or other Sustainable Development Goals."
)

def is_query_relevant(query: str) -> bool:
    result = guardrail_chain.invoke({"question": query}).strip().upper()
    return result == "RELEVANT"

# Test guardrail
print("Guardrail tests:")
print(f"  'What is the extreme poverty rate?' -> {is_query_relevant('What is the extreme poverty rate?')}")
print(f"  'Who won the World Cup 2022?'     -> {is_query_relevant('Who won the World Cup 2022?')}")

Guardrail tests:
  'What is the extreme poverty rate?' -> True
  'Who won the World Cup 2022?'     -> False


In [ ]:
# 3.3  RAG System Prompt & Conversational RAG Chain

SYSTEM_PROMPT = (
    "You are a knowledgeable assistant specialised in the UN Secretary‑General SDG Progress Report 2024.\n"
    "Answer questions accurately and concisely using ONLY the provided context from the report. If the context contains partial information, you may combine facts from different chunks as long as you do not invent anything.\n"
    "If the context does not contain enough information, say so honestly.\n"
    "Always mention the chapter or page number when available in the context.\n"
    "Use formal but accessible language.\n\n"
    "Context from the UN Secretary‑General SDG Progress Report 2024:\n{context}\n\n"
    "Conversation history:\n{chat_history}"
)

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(
    "Given the following conversation and a follow-up question, "
    "rephrase the follow-up question as a standalone question that includes all necessary context.\n\n"
    "Chat history:\n{chat_history}\n\n"
    "Follow-up question: {question}\n\n"
    "Standalone question:"
)

import warnings
warnings.filterwarnings("ignore")

# ConversationBufferWindowMemory: retains last 5 turns to prevent
# context window overflow while enabling multi-turn follow-up questions.
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
    k=5,
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=compression_retriever,
    memory=memory,
    condense_question_prompt=CONDENSE_QUESTION_PROMPT,
    combine_docs_chain_kwargs={
        "prompt": ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT),
            HumanMessagePromptTemplate.from_template("{question}")
        ])
    },
    return_source_documents=True,
    verbose=False,
)

print("Conversational RAG chain ready.")

Conversational RAG chain ready.


In [ ]:
# 3.4  Chat Handler

def chat(query: str) -> str:
    # Step 1: Guardrail — refuse off-topic queries
    if not is_query_relevant(query):
        return OUT_OF_SCOPE_REPLY

    # Step 2: RAG chain
    result = qa_chain.invoke({"question": query})
    answer = result["answer"]

    # Step 3: Append source citations
    sources = set()
    detected_goal = get_goal_for_query(query)

    for doc in result.get("source_documents", []):
        gl = doc.metadata.get("goal", "")
        pg = doc.metadata.get("page_label", doc.metadata.get("page", ""))
        if detected_goal and gl != detected_goal:
            continue

        if gl or pg:
            sources.add(f"{gl} (p.{pg})" if gl and pg else gl or f"p.{pg}")

    if sources:
        answer += f"\n\nSources: {', '.join(sorted(sources))}"

    return answer

print("Chat handler ready.")

Chat handler ready.


---

## **In-notebook Chat Interface (For Demo)**


In [ ]:
# ==========================
# IN-NOTEBOOK CHAT INTERFACE
# ==========================
import ipywidgets as widgets
from IPython.display import display

memory.clear()

style = """
<style>
.chat-container { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; }
.chat-header { background: #1a1a2e; color: white; padding: 14px 18px; border-radius: 12px 12px 0 0; }
.chat-messages { background: #f5f7fa; border: 1px solid #e2e8f0; border-top: none; height: 400px; overflow-y: auto; padding: 16px; }
.user-bubble { background: #1a1a2e; color: white; padding: 10px 14px; border-radius: 18px 18px 4px 18px; max-width: 75%; margin-left: auto; margin-bottom: 12px; width: fit-content; }
.bot-bubble { background: white; color: #1e293b; padding: 10px 14px; border-radius: 18px 18px 18px 4px; max-width: 85%; margin-right: auto; margin-bottom: 12px; border: 1px solid #e2e8f0; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }
.input-area { display: flex; gap: 8px; padding: 12px; border: 1px solid #e2e8f0; border-top: none; border-radius: 0 0 12px 12px; background: white; }
.input-area input { flex: 1; padding: 10px 14px; border: 1px solid #cbd5e1; border-radius: 24px; font-size: 14px; outline: none; }
.input-area button { padding: 10px 18px; border: none; border-radius: 24px; font-weight: 500; cursor: pointer; }
.btn-send { background: #1a1a2e; color: white; }
.btn-clear { background: #f1f5f9; color: #334155; border: 1px solid #cbd5e1; }
</style>
"""

title = widgets.HTML(f"{style}<div class='chat-header'><b>🌐 UN SDG Progress 2024</b> · RAG Assistant</div>")
chat_box = widgets.HTML(value="<div class='chat-messages'><div style='color:#94a3b8;text-align:center;padding-top:160px;'>Ask a question about the report…</div></div>")
input_text = widgets.Text(placeholder="Type your question…", layout=widgets.Layout(width="auto"))
send_btn = widgets.Button(description="Send", button_style="primary", layout=widgets.Layout(width="80px"))
clear_btn = widgets.Button(description="Clear", button_style="warning", layout=widgets.Layout(width="80px"))

messages = []

def render():
    html = "<div class='chat-messages'>"
    if not messages:
        html += "<div style='color:#94a3b8;text-align:center;padding-top:160px;'>Ask a question about the report…</div>"
    else:
        for role, txt in messages:
            escaped = txt.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;").replace("\n","<br>")
            if role == "user":
                html += f"<div class='user-bubble'>{escaped}</div>"
            else:
                html += f"<div class='bot-bubble'><b>🤖</b> {escaped}</div>"
    html += "</div>"
    chat_box.value = html

def on_send(b=None):
    q = input_text.value.strip()
    if not q: return
    input_text.value = ""
    messages.append(("user", q))
    # Temporary "Thinking..." bubble
    think_idx = len(messages)
    messages.append(("bot", "Thinking..."))
    render()
    try:
        resp = chat(q)
    except Exception as e:
        resp = f"[Error: {e}]"
    # Replace thinking bubble with actual response
    messages[think_idx] = ("bot", resp)
    render()

def on_clear(b):
    messages.clear()
    memory.clear()
    render()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
input_text.on_submit(on_send)

display(widgets.VBox([title, chat_box, widgets.HBox([input_text, send_btn, clear_btn], layout=widgets.Layout(padding="12px"))]))

###Test Queries

1. What is the overall progress on the SDG targets according to the 2024 report?
2. How has extreme poverty changed since 2015?
3. How has COVID-19 affected global hunger?
4. What are the trends for HIV, TB, and malaria?
5. What progress has been made on maternal health?
6. What does the report say about climate finance?
7. What share of women are in leadership positions globally?
8. Who won the 2022 FIFA World Cup?
